# Dependencies

In [1]:
import tqdm as notebook_tqdm
from datasets import load_dataset
from dotenv import load_dotenv
from typing import Callable, Any
from pathlib import Path
import torch as T
import numpy as np
import string
import os

# Setup

In [2]:
# Change print width
T.set_printoptions(linewidth=1000)
np.set_printoptions(linewidth=1000)

# Data
Importing the data

In [3]:
env_path = os.path.join(os.getcwd(), ".env.secrets")
load_dotenv(env_path)

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

## Data pre-processing
Preprocessing the data allows us to collect as much useful information as possible. Considering that our goal is to analyze the semantic meaning of individual words and not sentences, we made some specific design choices in the preprocessing. Both of our operations were chosen on the basis that our representations do not attempt to represent sentiment, inflection, grammatical usage, or any other contextualized differences between word representations. Instead, we focus on representing and comparing the words in isolation. Therefore, we perform the following operations:

- Remove punctuation: The inclusion of punctuation would be misleading since it is a tool for expressing meaning in the context of a sentence. For instance a question, a pause, an exclamation, and so on. They are grammatical tools which do not alter the meaning of the word they are attached to. For instance, "apple!", "apple?", "apple,", and "apple." all have the word apple which we aim to represent, not the inflection of the word.

- Convert to lowercase: This keeps all words equal before their representation, removing possible differences in representation for capitalized words such as "Apple" and "apple".

In [13]:
def pre_process_data(data: T.utils.data.Dataset):
    # We clean the words in the dataset a bit, this is to ensure that all words present
    # in the vocabulary are in a predictable format
    to_remove = string.punctuation + "“”‘’"
    table = str.maketrans("", "", to_remove)  # Create a translation table
    big_blob = " ".join(data["text"])
    # Translate the blob using the table and convert to lower
    cleaned_blob = big_blob.translate(table).lower()
    return sorted(list(set(cleaned_blob.split())))

# Setup Training Data

In [ ]:
class Tokenizer():
    def __init__(self, ds, num_max_sentences, window_size, PAD_TOKEN):
        self.ds = ds
        self.num_max_sentences = num_max_sentences
        self.window_size = window_size
        self.vocabulary = self.create_vocabulary(ds)
        self.PAD_TOKEN = PAD_TOKEN
        self.PAD_INDEX = len(self.vocabulary)

        self.word_to_index = {word: i for i, word in enumerate(self.vocabulary)}
        self.word_to_index[PAD_TOKEN] = self.PAD_INDEX

        self.index_to_word = {i: word for i, word in enumerate(self.vocabulary)}
        self.index_to_word[self.PAD_INDEX] = PAD_TOKEN

    def get_one_hot_encoding(self, word)

    def create_vocabulary(self, data: T.utils.data.Dataset):
        # We clean the words in the dataset a bit, this is to ensure that all words present
        # in the vocabulary are in a predictable format
        to_remove = string.punctuation + "“”‘’"
        table = str.maketrans("", "", to_remove)  # Create a translation table
        big_blob = " ".join(data["text"])
        # Translate the blob using the table and convert to lower
        cleaned_blob = big_blob.translate(table).lower()
        return sorted(list(set(cleaned_blob.split())))
    
    def preprocess_sentence(self, sentence: str) -> list[str]:                                                       
        to_remove = string.punctuation + "“”‘’"                                                                    
        table = str.maketrans("", "", to_remove)                                                                     
        return sentence.translate(table).lower().split()

    def create_CBOW_pairs(self, add_padding=False):
        pairs = []
        unk = self.word_to_index.get("unk")
        print(f"unk: {unk}")
        for sentence in (self.ds["text"][:self.num_max_sentences]):
            if len(sentence) < 1:
                continue
            ids = [self.word_to_index.get(w, unk) for w in self.preprocess_sentence(sentence)]
            for center in range(len(ids)):
                # collect left and right context, skip center
                context = []
                for offset in range(-self.window_size, self.window_size + 1):
                    if offset == 0:
                        continue
                    pos = center + offset
                    if 0 <= pos < len(ids):
                        context.append(ids[pos])
                if len(context) == 0:
                    continue
                if add_padding and len(context) < (2*self.window_size):
                    dn = 2*self.window_size - len(context)
                    context = [self.PAD_INDEX] * dn + context
                target = ids[center]
                pairs.append((context, target))
        return pairs


In [83]:
FN = "/home/enarde/Documents/Design-Of-AI-Systems/miniproject/tokenizer.pt"
max_sentences = 5
window_size = 2
PAD_TOKEN = "<PAD>"

file_path = Path(FN)
if file_path.exists():
    tokenizer = T.load(file_path, weights_only=False)
else:
    tokenizer = Tokenizer(x_train, max_sentences, window_size, PAD_TOKEN)
    T.save(tokenizer, file_path)

cbow_pairs = tokenizer.create_CBOW_pairs(add_padding=True)
print(f"{len(cbow_pairs)} CBOW pairs created from {max_sentences} sentences with window size: {window_size}")
print(f"Pad Token: {PAD_TOKEN}, Pad Index: {tokenizer.word_to_index[PAD_TOKEN]}")
for (context, target) in cbow_pairs:
    print(f"Target: {target}")
    print(f"Target decoded: {tokenizer.index_to_word[target]}")
    print(f"Context: {context}\n")
    print(f"Context decoded: {[tokenizer.index_to_word[index] for index in context]}")


unk: 209335
192 CBOW pairs created from 5 sentences with window size: 2
Pad Token: <PAD>, Pad Index: 226921
Target: 211512
Target decoded: valkyria
Context: [226921, 226921, 41012, 95910]

Context decoded: ['<PAD>', '<PAD>', 'chronicles', 'iii']
Target: 41012
Target decoded: chronicles
Context: [226921, 226921, 211512, 95910]

Context decoded: ['<PAD>', '<PAD>', 'valkyria', 'iii']
Target: 95910
Target decoded: iii
Context: [226921, 226921, 211512, 41012]

Context decoded: ['<PAD>', '<PAD>', 'valkyria', 'chronicles']
Target: 179709
Target decoded: senjō
Context: [226921, 226921, 140614, 211512]

Context decoded: ['<PAD>', '<PAD>', 'no', 'valkyria']
Target: 140614
Target decoded: no
Context: [226921, 179709, 211512, 3072]

Context decoded: ['<PAD>', 'senjō', 'valkyria', '3']
Target: 211512
Target decoded: valkyria
Context: [179709, 140614, 3072, 209335]

Context decoded: ['senjō', 'no', '3', 'unk']
Target: 3072
Target decoded: 3
Context: [140614, 211512, 209335, 41012]

Context decoded: 

In [84]:
class DataLoader():
    def __init__(
        self, 
        data, 
        batch_size,
        shuffle=False
    ):
        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self._indices = None

    def __iter__(self):
        indices = list(range(len(self.data)))
        if self.shuffle:
            np.random.shuffle(indices)
        self._indices =  indices
        self._pos = 0
        return self

    def __next__(self):                                                                                          
        if self._pos >= len(self.data):                                                                          
            raise StopIteration        
        batch = [self.data[i] for i in self._indices[self._pos : self._pos + self.batch_size]]                             
        self._pos += self.batch_size                                             
        return batch
    
dataloader = DataLoader(cbow_pairs, batch_size=32)

for batch in dataloader:
    for (context, target) in batch:
        print(f"Target: {target}")
        print(f"Context: {context}\n")

Target: 211512
Context: [226921, 226921, 41012, 95910]

Target: 41012
Context: [226921, 226921, 211512, 95910]

Target: 95910
Context: [226921, 226921, 211512, 41012]

Target: 179709
Context: [226921, 226921, 140614, 211512]

Target: 140614
Context: [226921, 179709, 211512, 3072]

Target: 211512
Context: [179709, 140614, 3072, 209335]

Target: 3072
Context: [140614, 211512, 209335, 41012]

Target: 209335
Context: [211512, 3072, 41012, 101704]

Target: 41012
Context: [3072, 209335, 101704, 226818]

Target: 101704
Context: [209335, 41012, 226818, 117347]

Target: 226818
Context: [41012, 101704, 117347, 211512]

Target: 117347
Context: [101704, 226818, 211512, 143543]

Target: 211512
Context: [226818, 117347, 143543, 199910]

Target: 143543
Context: [117347, 211512, 199910, 22343]

Target: 199910
Context: [211512, 143543, 22343, 3072]

Target: 22343
Context: [143543, 199910, 3072, 44586]

Target: 3072
Context: [199910, 22343, 44586, 166664]

Target: 44586
Context: [22343, 3072, 166664, 20

# Creating the modeling tools
We chose to define our model in an object oriented way. The reason for this being that we aim to increase the readability of our code by relating python objects to mathematical objects. Hopefully, creating a successfull bridge between the theory and the implementation.

Each layer of our model can be modeled as a function.
$$
        \mathbf{\hat{y}} = f(\mathbf{v})
$$
This function can contain any arbitrary operation on the input vector given that it returns a Tensor object as a result. This effectively functions as an interface for clients to use.

In [14]:
class Layer():

    def forward(self, v: Any) -> T.Tensor:
        raise NotImplementedError(f"{self.__name__} has not implemented forward")
    

In [15]:
class Derivable(Layer):

    def __init__(self):
        super().__init__()
        self.dLdv: T.Tensor = None

        def compute_derivatives(self, dLdy: T.Tensor):
            """
            Compute relevant derivatives of this layer
            """

        def backward(self, dLdy: T.Tensor):
            """
            Calculates and sets derivatives in the layer
            """
        

In [16]:
class Trainable(Derivable):

    def __init__(self):
        super().__init__()
        self.dLdw: T.Tensor = None
        """
        Derivative (D) of the output y (Y) w.r.t the Weights (DW) of the object.
        """
        self.dLdv: T.Tensor = None
        """
        Derivative (D) of the output y (Y) w.r.t the Input (DV) of the object.
        """

    def update_weights(self, dLdy: T.Tensor) -> None:
        """
        Update the weights of the object using the upstream derivative which is the loss w.r.t. 
        the next layer activations which becomes the current layer activations 
        """
        raise NotImplementedError(
            f"{self.__name__} has not implemented update_weights"
        )

## One-hot-encoder
Assigns each word in a vocabulary a $n$ dimensional vector with a single element set to one (uniquely) where the words are sorted alphabetically and $n$ is the number of unique words

In [99]:
class OneHotEncoder(Layer):
    def __init__(self, vocabulary_size: int):
        self.dim = vocabulary_size

    def forward(self, token: int) -> T.Tensor:
        vec = T.zeros(self.dim)
        vec[token] = 1
        return vec
    

## Vectorized One-hot-encoder

In [100]:
class OneHotEncoderVectorized(Layer):
    def __init__(self, encoder: OneHotEncoder):
        self.encoder = encoder

    def forward(self, tokens: list[int]) -> T.Tensor:
        return T.stack([self.encoder.forward(token) for token in tokens])

In [101]:
FN = "/home/enarde/Documents/Design-Of-AI-Systems/miniproject/one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
    encoder = T.load(file_path, weights_only=False)
else:
    encoder = OneHotEncoder(len(tokenizer.vocabulary))
    T.save(encoder, file_path)

## Hierarchical Softmax
Softmax is a function that turns a vector of numbers $\mathbf{z} \in \mathbf{R}^n$ in to a vector of probilities $\mathbf{\hat{y}}$ s.t $\sum_i \hat{y}_i = 1$. The softmax function is defined as:

$$\mathbf{\hat{y}}_i = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

The gradients are calculated automatically during the backward pass and are set as follows:

$$
    \begin{align*}
        & \delta_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\
        & \frac{\partial \hat{y}_i}{\partial z_j} = \hat{y}_i(\delta_{ij} - \hat{y}_j) \\
        & \text{diag}(\mathbf{\hat{y}}) = \begin{bmatrix} \hat{y}_1 & 0 & \dots \\ 0 & \hat{y}_2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}
        & \mathbf{\hat{y}}\mathbf{\hat{y}}^T = \begin{bmatrix} \hat{y}_1^2 & \hat{y}_1\hat{y}_2 & \dots \\ \hat{y}_2\hat{y}_1 & \hat{y}_2^2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}\\
        & \frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T
    \end{align*}
$$

The single element version $\frac{\partial \hat{y}_i}{\partial z_j}$ of the derivative is included for clarity as it makes it easier to understand the full vector ($\mathbf{\hat{y}}$). 

In order to calculate the partial derivative of the loss given the input $\mathbf{z}$ to the activation layer we can apply the upstream partial derivative denoted $\frac{\partial{L}}{\partial{\mathbf{\hat{y}}}}$ using the chain rule:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \left(\frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = (\text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} - \mathbf{\hat{y}}\mathbf{\hat{y}}^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \mathbf{\hat{y}}\odot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} - \left(\mathbf{\hat{y}}^T \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}}\right)\mathbf{\hat{y}} \\
    \end{align*}
$$
                
Where the first transposition is required to match the dimensionalities since we are working with column vectors. Furthermore, the last line follows from simplifications of the diagonal matrix multiplied by a vector reducing to an elementwise scale of the upstream partial derivative and the second line results from the dot product reducing the vector multiplication to a scalar multiplication. Note that $\left(\text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T\right)^T = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T$ due to symmetry.

In [102]:
class HierarchicalSoftmax(Derivable):
    def __init__(self):
        super().__init__()
        self._cached_output = None

    def forward(self, z: T.Tensor) -> T.Tensor:
        # Exponentiate all elements of z, z are the logits, or the "raw" input.
        z_exp = T.exp(z)
        # Squeeze away the last element of y, this makes no difference mathematically.
        y = (z_exp / T.sum(z_exp, dim=0)).squeeze()

        self._cached_output = y
        return y
    
    def compute_gradients(self, dLdy: T.Tensor):
        return (self._cached_output * (dLdy - T.dot(self._cached_output, dLdy)))
    
    def backward(self, dLdy: T.Tensor):
        self.dLdv = self.compute_gradients(dLdy)

sm = HierarchicalSoftmax()
test_values = [
    T.tensor([1.0, 0.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 1.0]).reshape(-1, 1),
]

for v in test_values:
    out = sm.forward(v)
    print(
        f"""Output:\n{out.numpy()},
        """
    )

Output:
[0.4753669 0.1748777 0.1748777 0.1748777],
        
Output:
[0.3655293  0.3655293  0.13447072 0.13447072],
        
Output:
[0.29692274 0.29692274 0.29692274 0.10923178],
        
Output:
[0.25 0.25 0.25 0.25],
        


## Linear layer
The object `LinearLayer2D` performs the following operation on an input vector $\mathbf{v}$:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W} + \mathbf{b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

Where $\mathbf{b}$ is the bias term, $\mathbf{W}$ is the weights, and $\mathbf{\sigma}$ is the activation function, $\mathbf{z}$ is just an intermediary step used for convenience.

We have, however, modified this operation slightly so that we can remove the separate bias term and include it directly in an expanded weight matrix $\mathbf{W_b}$. This is achieved by adding a row filled with ones to $\mathbf{W}$. So the operation actually becomes:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W_b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

The gradients are calculated as following during the backward pass:

**$\mathbf{\hat{y}}$ w.r.t the input vector $\mathbf{v}$**
$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} \\

        & \frac{\partial \mathbf{z_i}}{\partial \mathbf{v_j}} = \mathbf{W_b}_{ji} \\

        & \frac{\partial \mathbf{z}}{\partial \mathbf{v}} = \begin{bmatrix} \frac{\partial z_1}{\partial v_1} & \frac{\partial z_1}{\partial v_2} & \dots & \frac{\partial z_1}{\partial v_N} \\ \frac{\partial z_2}{\partial v_1} & \frac{\partial z_2}{\partial v_2} & \dots & \frac{\partial z_2}{\partial v_N} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial z_M}{\partial v_1} & \frac{\partial z_M}{\partial v_2} & \dots & \frac{\partial z_M}{\partial v_N} \end{bmatrix} = \mathbf{W_b}^T \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \text{ is taken from the activation function with} \space \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \space \text{applied} \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}} = \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \mathbf{W_b}^T        
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{v}}}$ with:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \left((\mathbf{W_b}^T)^T \cdot \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\right)^T \right) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \mathbf{W_b} \cdot \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \mathbf{W_b} \cdot \frac{\partial{L}}{\partial{\mathbf{z}}} \\
    \end{align*}
$$

**$\mathbf{\hat{y}}$ w.r.t the weight matrix $\mathbf{W_b}$**

$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} + 1 \cdot \mathbf{W_b}_{(n+1)i} \\

        & \frac{\partial z_i}{\partial \mathbf{W_b}_{kj}} = \begin{cases} v_k & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\

        & \frac{\partial \mathbf{z}}{\partial \mathbf{W_b}} = \left[ \quad \begin{bmatrix} v_1 & 0 & \dots & 0 \\ v_2 & 0 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ v_N & 0 & \dots & 0 \\ 1 & 0 & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \begin{bmatrix} 0 & v_1 & \dots & 0 \\ 0 & v_2 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & v_N & \dots & 0 \\ 0 & 1 & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \dots \quad \Bigg| \quad \begin{bmatrix} 0 & 0 & \dots & v_1 \\ 0 & 0 & \dots & v_2 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & v_N \\ 0 & 0 & \dots & 1 \end{bmatrix} \quad \right]
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ where $\frac{\partial{L}}{\partial{\mathbf{z}}}$ is our upstream gradient. We can do this by applying the chain rule element-wise:
$$
\begin{align*}
& \frac{\partial L}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot \frac{\partial z_j}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot v_{b,i}
\end{align*}
$$

This is equivalent to the outer product:
$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \\ 1 \end{bmatrix} \begin{bmatrix} \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} 
        v_1 \frac{\partial L}{\partial z_1} & v_1 \frac{\partial L}{\partial z_2} & \dots & v_1 \frac{\partial L}{\partial z_M} \\ 
        v_2 \frac{\partial L}{\partial z_1} & v_2 \frac{\partial L}{\partial z_2} & \dots & v_2 \frac{\partial L}{\partial z_M} \\ 
        \vdots & \vdots & \ddots & \vdots \\ 
        v_N \frac{\partial L}{\partial z_1} & v_N \frac{\partial L}{\partial z_2} & \dots & v_N \frac{\partial L}{\partial z_M} \\ 
        \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} 
        \end{bmatrix} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \otimes \frac{\partial L}{\partial \mathbf{z}}
    \end{align*}
$$

<!-- Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ where $\frac{\partial{L}}{\partial{\mathbf{z}}}$ is our upstream gradient which can contain the application of an activation or not. This becomes:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \left(\frac{\partial \mathbf{z}}{\partial \mathbf{W_b}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{z}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} v_{b,1} \\ v_{b,2} \\ \vdots \\ v_{b,N} \\ 1 \end{bmatrix} \begin{bmatrix} \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix} = \begin{bmatrix} v_{b,1} \frac{\partial L}{\partial z_1} & v_{b,1} \frac{\partial L}{\partial z_2} & \dots & v_{b,1} \frac{\partial L}{\partial z_M} \\ v_{b,2} \frac{\partial L}{\partial z_1} & v_{b,2} \frac{\partial L}{\partial z_2} & \dots & v_{b,2} \frac{\partial L}{\partial z_M} \\ \vdots & \vdots & \ddots & \vdots \\ v_{b,N} \frac{\partial L}{\partial z_1} & v_{b,N} \frac{\partial L}{\partial z_2} & \dots & v_{b,N} \frac{\partial L}{\partial z_M} \\ \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix}
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ with:

$$
    \begin{align*}
        & \frac{\partial L}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot v_{b,i} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \otimes \frac{\partial L}{\partial \mathbf{z}} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T
    \end{align*}
$$ -->

In [103]:
class LinearLayer2D(Trainable):
    def __init__(
        self,
        dim_in: int,
        dim_out: int,
        activation: Derivable,
    ):
        self.dim_in = dim_in
        self.dim_out = dim_out
        # Add bias dim
        weight_dims = (dim_in + 1, dim_out)
        # Initialize weights
        self.weights = T.ones(
            size=weight_dims,
        )
        self.activation = activation
        self._cached_input = None

    def __add_bias(self, v: T.Tensor) -> T.Tensor:
        # add 1 to the last dim and fill that with 1s to create a single
        # weight matrix that contains the bias term.
        bias = T.ones((1,))
        vec_plus_bias = T.cat((v, bias), dim=0)
        return vec_plus_bias

    def compute_derivatives(self, dLdy: T.Tensor):        
        # Create the derivatives, if the linear layer is a pure linear layer
        # there is no activation function derivative to take into account
        if self.activation is not None:
            dLdz = self.activation.dLdv
            dLdv = self.weights @ dLdz

            dLdw = T.outer(self._cached_input, dLdz)

            derivatives = (dLdv[:-1],dLdw)
        else:
            print(f"weights shape: {self.weights.shape}, dLdy shape: {dLdy.shape}")
            dLdv = self.weights @ dLdy
            dLdw = T.outer(self._cached_input, dLdy)

            derivatives = (dLdv[:-1],dLdw)

        return derivatives
    
    def update_weights(self, dLdy: T.Tensor) -> None:
        if self._cached_input is None:
            raise RuntimeError("Perform a forward pass before trying to update weights")
        
        if self.activation is not None:
            self.activation.backward(dLdy)

        dLdv, dLdw = self.compute_derivatives(dLdy)
        self.dLdw = dLdw
        self.dLdv = dLdv
        self.weights -= dLdw

    def forward(self, v: T.Tensor) -> T.Tensor:

        vec_plus_bias = self.__add_bias(v)
        self._cached_input = vec_plus_bias

        z = vec_plus_bias @ self.weights

        # In the case of no activation, dont call self.activation.
        if self.activation is not None:
            y = self.activation.forward(
                (z)
            ).squeeze()
        else:
            y = z.squeeze()
        
        return y

test_values = [T.ones((10,), dtype=T.float32)]

ll = LinearLayer2D(10, 3, HierarchicalSoftmax())
for v in test_values:
    out = ll.forward(v)
    w = ll.weights
    print(
        f"""Input:\n{v.numpy()},
        \nOutput:\n{out.numpy()},
        \nWeights:\n{w.numpy()},
        """
    )

Input:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.],
        
Output:
[0.33333334 0.33333334 0.33333334],
        
Weights:
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]],
        


## Linear layer average mixer
This object is used to average the the result of many layers of the same type.

In [104]:
class LinearLayerAvgMixer(Trainable):

    def __init__(self, n: int, layer: LinearLayer2D):
        super().__init__()
        self.n = n
        self.layer = layer

    def forward(self, v: T.Tensor) -> T.Tensor:
        avg = T.zeros(self.layer.dim_out)
        for i in range(self.n):
            y = self.layer.forward(v[i]) 
            avg += y / self.n
            # if i == 0:
            #     self.dLdv = self.layer.dLdv / self.n
            #     self.dLdw = self.layer.dLdw / self.n
            # else:
            #     self.dLdv += self.layer.dLdv / self.n
            #     self.dLdw += self.layer.dLdw / self.n
        return avg
    
    # def compute_derivatives(self, dLdv: T.Tensor):        
    #     return self.layer.compute_derivatives(dLdv)
    
    def update_weights(self, dLdy: T.Tensor) -> None:
        self.layer.update_weights(dLdy)

## The Model object
This object holds all the layers and activation functions we have defined, it represents the implementation of the full function that produces and outputs $\mathbf{\hat{y}}$ from an input $\mathbf{v}$.

In [105]:
class Model:
    def __init__(self, layers: list[Trainable | Derivable | object]):
        self.layers = layers

    def forward(self, v: T.Tensor):
        """
        Performs a forward pass.
        """
        curr = v
        for l in self.layers:
            print(
                f"Input to layer {l}: {curr.numpy() if isinstance(curr,T.Tensor) else curr}, with shape {curr.numpy().shape if isinstance(curr,T.Tensor) else curr.shape}"
            )
            curr = l.forward(curr)
            print(
                f"Output from layer {l}: {curr.numpy()}, with shape {curr.numpy().shape}"
            )
        return curr

    def backpropogate(self, dLdy: T.Tensor) -> None:
        dLdy = dLdy # Start off by using the derivative of the loss w.r.t to the output
        for l in reversed(self.layers):
            # If the layer is trainable update its weights
            if isinstance(l, Trainable): 
                l.update_weights(dLdy)
            # Update the derivative of the next layer output w.r.t its input
            # (which is the derivative of the output of the current layer w.r.t to its input)
            if not isinstance(l, OneHotEncoderVectorized):
                dLdy = l.dLdv

## Binary cross entropy loss
The Binary cross entropy loss function is a loss function that is primarily used to compare probability vectors (vectors where elements sum to $1$) against binary vectors (where a single element is set to $1$ and there rest are set to $0$.) This is usally used in binary classification where a model predicts a probability of a sample having some label, for example, $cat$, $dog$, $fish$. In binary classification is only a single correct lable but to improve model performance we allow it to be kind of uncertain, so it can predict $90\%$ $dog$ , $5\%$ $cat$, $5\%$ $fish$.
$$
    L = -\big(y \log(\hat{y}) + (1 - y) \log(1 - \hat{y})\big)
$$


In [106]:
class BinaryCrossEntropyLoss():
    def __init__(self):
        self.loss = None
        self.dLdy = None

    def compute_loss(self, y_true: T.Tensor, y_hat: T.Tensor):
        # Log of 0 gives inf. Clamping to avoid crashes
        y_hat = T.clamp(y_hat, min=1e-7, max=1 - 1e-7)

        loss = -(y_true @ T.log(y_hat) + (1-y_true) @ (T.log(1-y_hat)))
        self.loss = loss

        dLdy_hat = ((-y_true) / y_hat) + ((1-y_true) / (1-y_hat))
        self.dLdy = dLdy_hat
        return loss
        

## Training harness
An object that trains a Model object using stochastic gradient descent (SGD)
 
```py
cel = CrossEntropyLoss()
mod = Word2Vec():
for epoch in nr_epochs:
    for b in get_batch(training_data):
        y_hat = mod(b)
        loss = cel(y_hat, y_true)
        dLdy = cel.dLdy
        mod.backpropogate(dLdy)
        ...
    ...
```

In [107]:
def train(model, dataloader, loss_fn, n_epochs):
    for epoch in range(n_epochs):                                                                                
        epoch_loss = 0                                                                                           
                                                                                                                
        for batch in dataloader:                                                                                 
            for (context, target) in batch:
                # 1. Encode inputs
                context_encoded = ...   # one-hot or index lookup                                                
                target_encoded = ...    # one-hot                                                                
                                                                                                                
                # 2. Forward pass                                                                                
                y_hat = model.forward(context_encoded)
                y_hat = sm.forward(y_hat)  # softmax                                                             

                # 3. Compute loss                                                                                
                loss = loss_fn.compute_loss(target_encoded, y_hat)
                epoch_loss += loss                                                                               
                
                # 4. Backward pass
                sm.backward(loss_fn.dLdy)
                model.backpropogate(sm.dLdv)                                                                     
                                                                                                                
        print(f"Epoch {epoch}: loss={epoch_loss:.4f}")

# Creating the Word2Vec model

In [109]:
INPUT_DIM = encoder.dim
OUTPUT_DIM = encoder.dim
LATENT_DIM = 10

word2vecCBOW = Model(
    [
        OneHotEncoderVectorized(
            encoder=encoder
        ),  # Use the saved encoder!
        LinearLayerAvgMixer(
            n=3,
            layer=LinearLayer2D(
                dim_in=INPUT_DIM, 
                dim_out=LATENT_DIM, 
                activation=None
            ),
        ),
        LinearLayer2D(
            dim_in=LATENT_DIM, 
            dim_out=OUTPUT_DIM, 
            activation=HierarchicalSoftmax()
        ),
    ]
)

test_values = np.array([1, 2, 3])
gold_label = [4]
gold_label_encoded = encoder.forward(gold_label)
result = word2vecCBOW.forward(test_values)

ce = BinaryCrossEntropyLoss()
loss = ce.compute_loss(gold_label_encoded, result)
print(f"Loss: {loss}, dldy: {ce.dLdy}")
word2vecCBOW.backpropogate(ce.dLdy)

Input to layer <__main__.OneHotEncoderVectorized object at 0x7117c1286bd0>: [1 2 3], with shape (3,)
Output from layer <__main__.OneHotEncoderVectorized object at 0x7117c1286bd0>: [[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]], with shape (3, 226921)
Input to layer <__main__.LinearLayerAvgMixer object at 0x7117c1286060>: [[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]], with shape (3, 226921)
Output from layer <__main__.LinearLayerAvgMixer object at 0x7117c1286060>: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2.], with shape (10,)
Input to layer <__main__.LinearLayer2D object at 0x7117c1286c60>: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2.], with shape (10,)
Output from layer <__main__.LinearLayer2D object at 0x7117c1286c60>: [4.40682e-06 4.40682e-06 4.40682e-06 ... 4.40682e-06 4.40682e-06 4.40682e-06], with shape (226921,)
Loss: 13.333243370056152, dldy: tensor([1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000])
weights shape: torch.Size([226922, 10]), dL

# Training, Validation, Testing

## Continous bag of words

## Skip-gram

In [ ]:
#TODO implement regularization so we can select a lambda param during validation
#TODO try different output sizes on the linear layer

# Visualizing results

## PCA
...

## UMAP
...

In [51]:
#TODO visualize results using UMAP and PCA